In [ ]:
# Paths are resolved relative to the repository, or to $PUBO_DATA if set.
# The original research code used absolute paths on the authors' machine;
# only the root has been parameterised, the directory layout is unchanged.
import os
DATA_ROOT = os.environ.get('PUBO_DATA',
                           os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'legacy'))
EXTERNAL_ROOT = os.environ.get('PUBO_EXTERNAL', DATA_ROOT)  # data not shipped with the repo


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import os
from IPython.display import clear_output
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv

In [ ]:
### Solver ###
class Hopfield_class():
    #Initial parameters
    def __init__(self, sigma, beta_0, alpha, dt, K, N, MAX_ITER, attempt_num, device = 'cuda'):
        self.MAX_ITER = MAX_ITER
        self.device = device
        self.alpha = alpha
        self.K = torch.tensor(K, dtype = torch.float32, device = self.device)
        self.dt = dt
        self.N = N
        self.attempt_num = attempt_num
        self.s_data = torch.zeros((self.N, self.MAX_ITER), dtype=torch.float32, device=torch.device('cuda'))
        self.u_data = torch.zeros((self.N, self.MAX_ITER), dtype=torch.float32, device=torch.device('cuda'))
        self.beta_data = torch.zeros((self.MAX_ITER), dtype=torch.float32, device=torch.device('cuda'))
        self.random_n=np.random.randint(self.attempt_num)
        self.sigma = sigma
        self.beta_0 = beta_0
        
    ### Solver ###
    def Hopfield_step(self, u, beta):  
        s = torch.tanh(u / beta)

        tensor_sum = torch.einsum('ijk,js,ks->is', (self.K,s,s))
        diff = (- u - 0.5 * tensor_sum)
        u = u + self.dt * diff
        beta = beta * self.alpha * (torch.sum(torch.abs(diff), axis=0, keepdim=True) < 0.001).float() + beta * (1 - (torch.sum(torch.abs(diff), axis=0, keepdim=True)<0.001).float())
        return s, u, beta
    
    def polynomial_expression(self, x):
        Energy_sums = torch.einsum('ijk,is,js,ks->s',(self.K,x,x,x))
        return Energy_sums

    def Hopfield_eval(self):
        u = self.sigma * torch.rand((self.N, self.attempt_num), dtype=torch.float32, device=torch.device('cuda')) - self.sigma / 2 * torch.ones((self.N, self.attempt_num), dtype=torch.float32, device=torch.device('cuda'))
        s, u, beta = self.Hopfield_step(u, self.beta_0)
#         beta = self.beta_0
        for m in range(self.MAX_ITER):
            s, u, beta = self.Hopfield_step(u, beta)
            self.s_data[:,m] = s[:,self.random_n]
            self.u_data[:,m] = u[:,self.random_n]
            self.beta_data[m] = beta[0,self.random_n]
        self.lastcuts=self.polynomial_expression(torch.sign(s))
        return self.lastcuts, torch.sign(s)
  

In [ ]:
class HopfieldInterface(mli.Interface):
        def __init__(self, hopfield, attempt_num, clip_percents): 
            super(HopfieldInterface, self).__init__()
            self.hopfield=hopfield
            self.attempt_num = attempt_num
            self.hopfield.Hopfield_eval()

            self.mean = np.mean(np.array(hopfield.lastcuts.cpu()))
            self.scaleen = np.min(np.array(hopfield.lastcuts.cpu()))
            self.minimum = np.min(np.array(hopfield.lastcuts.cpu()))



        def get_sim_params(self, packed_params):
            num_single = 4
            logsig, logdt, logbeta, logalpha  = packed_params[:num_single]

            sigma = 10**logsig
            dt = 10**logdt
            beta_0 = 10**logbeta
            alpha = 10** logalpha
            return sigma, dt, beta_0, alpha


        def get_next_cost_dict(self, params_dict):
            params = params_dict['params']

            sigma, dt, beta_0, alpha = self.get_sim_params(params)

            print('\nHopfield:\nsigma={}\ndt={}\nbeta={}\nalpha={}\n'.format(sigma, dt, beta_0, alpha))

            self.hopfield.sigma = sigma
            self.hopfield.dt = dt
            self.hopfield.beta_0 = beta_0
            self.hopfield.alpha = alpha
            
            self.hopfield.Hopfield_eval()
            cost_perc = min([np.mean(np.array(self.hopfield.lastcuts.cpu())) / np.abs(self.mean),1]) #np.min(self.TGD.lastcuts)
            cost_min = min([np.min(np.array(self.hopfield.lastcuts.cpu())) / np.abs(self.scaleen), 1]) 
            cost=cost_min + 0.5*cost_perc
            uncer = 0
            bad = False
            print('Initial perc', self.mean)
#             target_perc=self.mean
#             target_scale=self.scale_en
            print('Final perc', np.mean(np.array(self.hopfield.lastcuts.cpu())))
            print('Final energy', np.min(np.array(self.hopfield.lastcuts.cpu())))
            cost_dict = {'cost':cost, 'uncer':uncer, 'bad':bad}
            return cost_dict

In [ ]:
dt = 0.02
MAX_ITER = 2500
attempt_num=2000
#Initialisation
beta_0 = 10.0
sigma = 10.0
alpha = 0.8
for i in range(0,10):
    psize=int(10*(i+1))
    for number in range(0,10):
        sigma, dt, beta, alpha = np.load(DATA_ROOT+'/sk_symmetric/Hopfield_each/'+str(psize)+'_'+str(number)+'_params.npy')

    #     for number in range(0,10):
        foldername = str(psize)+'_'+str(number)
        K= torch.tensor(np.load(DATA_ROOT+'/sk_symmetric/'+foldername+'.npy'), dtype=torch.float64, device=torch.device('cuda'))
    #K = K_imported.reshape(100,100,100)
        unity=torch.ones((K.size(0)), dtype=torch.float32, device='cuda')
        N = torch.Tensor.size(K)[0]


        #torch.random.seed(number)
        #u = sigma * torch.rand((N, attempt_num), dtype=torch.float32, device=torch.device('cuda')) - sigma / 2 * torch.ones((N, attempt_num), dtype=torch.float32, device=torch.device('cuda'))
        #s = torch.tanh(u/beta).float32()

        #s = solution*0.999
        #u = beta*np.arctanh(s)

        #Record
        #Run
    #         if number==0:

        clip_percents=50
        population_size=7
        percentile = 50

        hopfield = Hopfield_class( sigma, beta, alpha, dt, K, N, MAX_ITER, attempt_num, device = 'cuda')
        interface = HopfieldInterface(hopfield, attempt_num, percentile)

        first_params = np.array([np.log10(hopfield.sigma), np.log10(hopfield.dt), np.log10(hopfield.beta_0), np.log10(hopfield.alpha)])
        min_boundary = np.array([-4, -4,-4,-4])
        max_boundary = np.array([ 4, 0, 4, 4])
        num_params = 4
        config = dict(controller_type='neural_net', 
                      training_type = 'differential_evolution',
                      first_params = first_params,
                      min_boundary = min_boundary, 
                      max_boundary = max_boundary,
                      num_params = num_params,
                      trust_region = 0.5,#0.5
                      max_num_runs = 1500,
                      num_training_runs = 2 * num_params,
                      log_level = 0, 
                      no_delay = False, 
                      max_num_runs_without_better_params = 150, 
                      #evolution_strategy='best1', 
                      population_size=10, 
                      mutation_scale = (0.5,1.0), 
                      cross_over_probability = 0.7, 
                      default_bad_cost = 1.05, 
                      #target_cost=-(8+np.percentile(np.array(TGD.lastcuts.cpu()), clip_percents))/(-np.min(np.array(TGD.lastcuts.cpu()))+np.percentile(np.array(TGD.lastcuts.cpu()), clip_percents)),
                      default_bad_uncertainty = 0.01,
                      cost_has_noise = True)

        controller = mlc.create_controller(interface, **config)
        controller.optimize()

        clear_output(True)

        print('Best cut found:')
        #print(-controller.best_cost*controller.interface.scale+controller.interface.perc)

        mlv.show_all_default_visualizations(controller)
        sigma=10**controller.best_params[0]
        dt=10**controller.best_params[1]
        beta=10**controller.best_params[2]
        alpha=10**controller.best_params[3]


        hopfield = Hopfield_class(sigma, beta, alpha, dt, K, N, MAX_ITER, attempt_num, device = 'cuda')
        energy_hop, spins = hopfield.Hopfield_eval()
        spins=spins.double()
        K_doub=K.double()
        energy_hop=torch.einsum('ijk,is,js,ks->s',(K_doub,spins,spins,spins))
        energy_hop = energy_hop.cpu().numpy()
        spins = spins.cpu().numpy()
    #         print(energy_hop.cpu().min())
        params=[sigma, dt, beta, alpha]
        plt.hist(energy_hop, bins = 100)
        plt.show()
        np.save(DATA_ROOT+'/sk_symmetric/Hopfield_each/'+ str(psize)+'_'+str(number)+'.npy', energy_hop)
        np.save(DATA_ROOT+'/sk_symmetric/Hopfield_each/'+ str(psize)+'_'+str(number)+'_params.npy', params)
        np.save(DATA_ROOT+'/sk_symmetric/Hopfield_each/'+ str(psize)+'_'+str(number)+'_bit.npy', spins)

    # # #         #Plot
    # #         time = torch.linspace(0.0, MAX_ITER * dt, MAX_ITER)

    # #         plt.figure(figsize=(20,10))
    # #         plt.subplot(311)
    # #         plt.plot(time,s_data.t().cpu())
    # #         plt.title('Hopfield Evolution')
    # #         plt.ylabel('s')

    # #         plt.subplot(312)
    # #         plt.plot(time,u_data.t().cpu())
    # #         plt.ylabel('u')

    # #         plt.subplot(313)
    # #         plt.plot(time,beta_data.cpu())
    # #         plt.ylabel('beta')
    # #         plt.xlabel('time (units)')

        plt.show()